In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

df1 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202505.csv")
df2 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202506.csv")
df3 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202507.csv")
df4 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202508.csv")
df5 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202509.csv")
df6 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202510.csv")
df7 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202511.csv")
df8 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202512.csv")
df9 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202601.csv")
df10 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202602.csv")
df11 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202603.csv")
df12 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202604.csv")
df13 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202605.csv")


/tmp/ipykernel_47617/3312040348.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202506.csv")
/tmp/ipykernel_47617/3312040348.py:11: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df9 = pd.read_csv("/content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/CRMLSSold202601.csv")


In [4]:
df_original = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13], ignore_index=True)
df = df_original.copy()

In [5]:
df_sf = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

In [6]:
df_sf["CloseDate"] = pd.to_datetime(df_sf["CloseDate"], errors="coerce")

#Features

In [7]:
location_features = [
    "Latitude",
    "Longitude",
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
]

property_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "GarageSpaces",
    "ParkingTotal",
]

lot_financial_features = [
    "AssociationFee",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
]

#Target

In [8]:
target = "ClosePrice"
date_col = "CloseDate"

all_features = location_features + property_features + lot_financial_features
required_columns = all_features + [target, date_col]

In [9]:
missing_columns = [col for col in required_columns if col not in df_sf.columns]
df_model = df_sf[required_columns].copy()

print("Input feature count:", len(all_features))
print("Selected dataset shape:", df_model.shape)

Input feature count: 21
Selected dataset shape: (141997, 23)


In [10]:
def build_time_split(data, x_months=12, date_col="CloseDate"):
    data = data.dropna(subset=[date_col]).copy()
    data["CloseMonth"] = data[date_col].dt.to_period("M")

    available_months = sorted(data["CloseMonth"].unique())
    test_month = available_months[-1]
    train_months = available_months[-(x_months + 1):-1]

    train = data[data["CloseMonth"].isin(train_months)].copy()
    test = data[data["CloseMonth"] == test_month].copy()

    return train, test, train_months, test_month


X_MONTHS = 12

train_df, test_df, train_months, test_month = build_time_split(
    df_model,
    x_months=X_MONTHS,
    date_col=date_col
)

print("Train months:", train_months[0], "to", train_months[-1])
print("Test month:", test_month)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train months: 2025-05 to 2026-04
Test month: 2026-05
Train shape: (129973, 24)
Test shape: (12024, 24)


# **Handle missing values**

In [11]:
train_missing = pd.DataFrame({
    "Missing Count": train_df[all_features].isna().sum(),
    "Missing %": (train_df[all_features].isna().mean() * 100).round(2),
    "Dtype": train_df[all_features].dtypes.astype(str),
}).sort_values("Missing %", ascending=False)

train_missing

,Missing Count,Missing %,Dtype
AssociationFee,37740,29.04,float64
HighSchoolDistrict,34957,26.90,object
MLSAreaMajor,18970,14.60,object
AttachedGarageYN,15774,12.14,object
Stories,13428,10.33,float64
ViewYN,11758,9.05,object
PoolPrivateYN,9932,7.64,object
GarageSpaces,5143,3.96,float64
LotSizeAcres,2230,1.72,float64
LotSizeSquareFeet,2229,1.71,float64


In [12]:
numeric_features = [
    "Latitude",
    "Longitude",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "GarageSpaces",
    "ParkingTotal",
    "AssociationFee",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
]

high_cardinality_categorical = [
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
]

low_cardinality_categorical = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
]

In [13]:
for frame in [train_df, test_df]:
    frame["ParkingTotal"] = frame["ParkingTotal"].where(
        frame["ParkingTotal"] >= 0,
        np.nan
    )

In [14]:
train_medians = train_df[numeric_features].median()

for col in numeric_features:
    if train_df[col].isna().any() or test_df[col].isna().any():
        train_df[f"{col}_missing"] = train_df[col].isna().astype(int)
        test_df[f"{col}_missing"] = test_df[col].isna().astype(int)

    train_df[col] = train_df[col].fillna(train_medians[col])
    test_df[col] = test_df[col].fillna(train_medians[col])

In [15]:
for col in high_cardinality_categorical + low_cardinality_categorical:
    train_df[col] = train_df[col].fillna("Missing").astype(str)
    test_df[col] = test_df[col].fillna("Missing").astype(str)

In [16]:
train_df["PostalCode"] = train_df["PostalCode"].astype(str)
test_df["PostalCode"] = test_df["PostalCode"].astype(str)

print("Remaining training-set missing values:")
print(train_df[all_features].isna().sum().sort_values(ascending=False).head())

print("\nRemaining test-set missing values:")
print(test_df[all_features].isna().sum().sort_values(ascending=False).head())


Remaining training-set missing values:
Latitude          0
Longitude         0
City              0
PostalCode        0
CountyOrParish    0
dtype: int64

Remaining test-set missing values:
Latitude          0
Longitude         0
City              0
PostalCode        0
CountyOrParish    0
dtype: int64


In [17]:
y_train = train_df[target].copy()
y_test = test_df[target].copy()

target_encoder = TargetEncoder(
    target_type="continuous",
    smooth="auto",
    random_state=42
)

train_high_encoded = target_encoder.fit_transform(
    train_df[high_cardinality_categorical],
    y_train
)
test_high_encoded = target_encoder.transform(
    test_df[high_cardinality_categorical]
)

train_high_encoded = pd.DataFrame(
    train_high_encoded,
    columns=[f"{col}_encoded" for col in high_cardinality_categorical],
    index=train_df.index
)
test_high_encoded = pd.DataFrame(
    test_high_encoded,
    columns=[f"{col}_encoded" for col in high_cardinality_categorical],
    index=test_df.index
)

In [18]:
onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)

train_low_encoded = onehot_encoder.fit_transform(
    train_df[low_cardinality_categorical]
)
test_low_encoded = onehot_encoder.transform(
    test_df[low_cardinality_categorical]
)

onehot_names = onehot_encoder.get_feature_names_out(
    low_cardinality_categorical
)

train_low_encoded = pd.DataFrame(
    train_low_encoded,
    columns=onehot_names,
    index=train_df.index
)
test_low_encoded = pd.DataFrame(
    test_low_encoded,
    columns=onehot_names,
    index=test_df.index
)

print("Target-encoded columns:", train_high_encoded.columns.tolist())
print("One-hot encoded columns:", onehot_names.tolist())

Target-encoded columns: ['City_encoded', 'PostalCode_encoded', 'CountyOrParish_encoded', 'MLSAreaMajor_encoded', 'HighSchoolDistrict_encoded']
One-hot encoded columns: ['ViewYN_Missing', 'ViewYN_True', 'PoolPrivateYN_Missing', 'PoolPrivateYN_True', 'AttachedGarageYN_Missing', 'AttachedGarageYN_True']


In [19]:
raw_numeric_features = numeric_features

missing_flag_features = [
    col for col in train_df.columns
    if col.endswith("_missing")
]

X_train = pd.concat(
    [
        train_df[raw_numeric_features + missing_flag_features],
        train_high_encoded,
        train_low_encoded,
    ],
    axis=1
)

X_test = pd.concat(
    [
        test_df[raw_numeric_features + missing_flag_features],
        test_high_encoded,
        test_low_encoded,
    ],
    axis=1
)

X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

non_numeric_train = X_train.select_dtypes(exclude="number").columns.tolist()

print("Final training feature count:", X_train.shape[1])
print("Non-numeric columns remaining:", non_numeric_train)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


Final training feature count: 36
Non-numeric columns remaining: []
X_train shape: (129973, 36)
X_test shape: (12024, 36)
y_train shape: (129973,)
y_test shape: (12024,)


In [20]:
train_export = X_train.copy()
train_export[target] = y_train.values
train_export[date_col] = train_df[date_col].values
train_export["split"] = "train"

test_export = X_test.copy()
test_export[target] = y_test.values
test_export[date_col] = test_df[date_col].values
test_export["split"] = "test"

train_export = train_export[train_export["ClosePrice"] > 0].copy()
test_export = test_export[test_export["ClosePrice"] > 0].copy()

cleaned_data = pd.concat(
    [train_export, test_export],
    ignore_index=True
)

In [21]:
train_df.to_csv("train_preprocessed.csv", index=False)
test_df.to_csv("test_preprocessed.csv", index=False)

print("train_preprocessed.csv:", train_df.shape)
print("test_preprocessed.csv:", test_df.shape)

train_export.to_csv("cleaned_train.csv", index=False)
test_export.to_csv("cleaned_test.csv", index=False)
cleaned_data.to_csv("cleaned_data.csv", index=False)

print("cleaned_train.csv:", train_export.shape)
print("cleaned_test.csv:", test_export.shape)
print("cleaned_data.csv:", cleaned_data.shape)

train_preprocessed.csv: (129973, 36)
test_preprocessed.csv: (12024, 36)
cleaned_train.csv: (129972, 39)
cleaned_test.csv: (12024, 39)
cleaned_data.csv: (141996, 39)
